# Adaptive Bead Pull Variations

This notebook runs the adaptive bead-pull calculation over all requested configurations: line counts 1-4, z-slice counts 1-3, and gap counts 1-4 ordered by gap-rank along the booster. The z-slice counts are interpreted per gap using the rules from `src.adaptive_bead_pull` (expressed in **absolute** z, i.e. z_abs = z_mirror - z_rel): gap 1 starts from the smallest z_abs, gap 2 from the largest z_abs, gap 3 from the smallest z_abs, and gap 4 from the 5th-smallest z_abs. Additional slices grow toward the opposite end of the gap in z_abs. It saves per-configuration figures under `figs/configs/` and comparison metrics/plots under `data/adaptive_bead_pull_variations/` and `figs/comparisons/`.


In [1]:
from pathlib import Path
import sys
import numpy as np

import src.adaptive_bead_pull as abp

In [2]:
REPO_ROOT = Path.cwd()
RESULTS_ROOT = REPO_ROOT / "data" / "adaptive_bead_pull_variations"
FIT_CACHE_DIR = RESULTS_ROOT / "fits"
FULL_CACHE_DIR = RESULTS_ROOT / "full_reference"
SLURM_DIR = RESULTS_ROOT / "slurm"
FIGS_ROOT = REPO_ROOT / "figs"
ET_VARIATIONS_PATH = REPO_ROOT / "data" / "ET_results_variations.npz"

LINE_COUNTS = abp.DEFAULT_LINE_COUNTS
Z_IX_OPTIONS = abp.DEFAULT_Z_IX_OPTIONS
GAP_COUNTS = abp.DEFAULT_GAP_COUNTS

N_BASELINE_MARGIN = abp.DEFAULT_N_BASELINE_MARGIN
FREQUENCY_INDICES = None  # keep None for the exact full calculation
MAKE_FIGURES = True

SUBMIT_SLURM = True
WAIT_FOR_SLURM = True
SLURM_PARTITION = "maxcpu"
SLURM_CPUS_PER_TASK = 40
SLURM_TIME_LIMIT = "04:00:00"
SLURM_MEMORY = "16G"
PYTHON_EXECUTABLE = sys.executable

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
FIGS_ROOT.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT, FIGS_ROOT

(PosixPath('/data/dust/user/salamana/madmax/spider-bead-pull/data/adaptive_bead_pull_variations'),
 PosixPath('/data/dust/user/salamana/madmax/spider-bead-pull/figs'))

## Gap Ordering

In [3]:
gaps = abp.discover_gap_files()
abp.print_gap_mapping(gaps)

assert len(gaps) == 4, f"Expected exactly 4 gap files, found {len(gaps)}"

gap 1: 2.0-4.0 mm, measurement_Al2O3_3mm_2025_09_19_18_59.hdf5, shape=(31, 19, 3, 1001)
gap 2: 16.0-18.0 mm, measurement_Al2O3_3mm_2025_09_23_17_15.hdf5, shape=(31, 19, 3, 1001)
gap 3: 22.0-24.0 mm, measurement_Al2O3_3mm_2025_09_23_08_18.hdf5, shape=(31, 19, 3, 1001)
gap 4: 35.0-42.0 mm, measurement_Al2O3_3mm_2025_09_21_05_54.hdf5, shape=(31, 19, 8, 1001)


## Automatic Cleanup For A Fresh Run

In [4]:
removed_stale_caches = abp.prune_stale_gap_caches(
    gaps,
    fit_cache_dir=FIT_CACHE_DIR,
    full_cache_dir=FULL_CACHE_DIR,
    line_counts=LINE_COUNTS,
    z_ix_options=Z_IX_OPTIONS,
)
removed_outputs = abp.reset_generated_outputs(
    result_root=RESULTS_ROOT,
    figs_root=FIGS_ROOT,
    keep_diagnostics=True,
)
removed_artifacts = []
if ET_VARIATIONS_PATH.exists():
    ET_VARIATIONS_PATH.unlink()
    removed_artifacts.append(ET_VARIATIONS_PATH)

print(f"removed stale caches: {len(removed_stale_caches)}")
print(f"removed generated outputs: {len(removed_outputs)}")
print(f"removed ET variation artifacts: {len(removed_artifacts)}")


removed stale caches: 16
removed generated outputs: 4
removed ET variation artifacts: 1


## One-Time Diagnostics

In [5]:
diagnostic_paths = abp.save_hg_sanity_figures(FIGS_ROOT)
diagnostic_paths

[PosixPath('/data/dust/user/salamana/madmax/spider-bead-pull/figs/diagnostics/hg_modes_abs.png'),
 PosixPath('/data/dust/user/salamana/madmax/spider-bead-pull/figs/diagnostics/hg_modes_real.png'),
 PosixPath('/data/dust/user/salamana/madmax/spider-bead-pull/figs/diagnostics/hg_modes_imag.png')]

## Slurm Fitting Cache

The fit cache has one file per `(line_count, gap)` pair. Each Slurm task fits the per-gap maximum three-slice selection used by this notebook, and the later assembly reuses the appropriate prefix for the 1-slice and 2-slice configurations.


In [6]:
complete, missing = abp.all_fit_caches_exist(
    gaps,
    line_counts=LINE_COUNTS,
    fit_cache_dir=FIT_CACHE_DIR,
    z_ix_options=Z_IX_OPTIONS,
)
print(f"fit cache complete before submit: {complete}")
print(f"missing fit caches: {len(missing)}")

submit_info = None
if (not complete) and SUBMIT_SLURM:
    submit_info = abp.submit_slurm_fit_array(
        repo_root=REPO_ROOT,
        gaps=gaps,
        fit_cache_dir=FIT_CACHE_DIR,
        slurm_dir=SLURM_DIR,
        line_counts=LINE_COUNTS,
        partition=SLURM_PARTITION,
        cpus_per_task=SLURM_CPUS_PER_TASK,
        time_limit=SLURM_TIME_LIMIT,
        mem=SLURM_MEMORY,
        python_executable=PYTHON_EXECUTABLE,
        z_ix_options=Z_IX_OPTIONS,
    )
    print(submit_info)
elif complete:
    print("All fit caches already exist; no Slurm submission needed.")
else:
    print("SUBMIT_SLURM is False. Submit manually or set SUBMIT_SLURM=True and rerun this cell.")

if WAIT_FOR_SLURM and (complete or SUBMIT_SLURM):
    abp.wait_for_fit_caches(
        gaps,
        fit_cache_dir=FIT_CACHE_DIR,
        line_counts=LINE_COUNTS,
        z_ix_options=Z_IX_OPTIONS,
        poll_seconds=60,
    )


fit cache complete before submit: False
missing fit caches: 16
{'submitted': True, 'job_id': '22649724', 'stdout': 'Submitted batch job 22649724', 'missing_before_submit': ['/data/dust/user/salamana/madmax/spider-bead-pull/data/adaptive_bead_pull_variations/fits/line_01_gap_01.pkl', '/data/dust/user/salamana/madmax/spider-bead-pull/data/adaptive_bead_pull_variations/fits/line_01_gap_02.pkl', '/data/dust/user/salamana/madmax/spider-bead-pull/data/adaptive_bead_pull_variations/fits/line_01_gap_03.pkl', '/data/dust/user/salamana/madmax/spider-bead-pull/data/adaptive_bead_pull_variations/fits/line_01_gap_04.pkl', '/data/dust/user/salamana/madmax/spider-bead-pull/data/adaptive_bead_pull_variations/fits/line_02_gap_01.pkl', '/data/dust/user/salamana/madmax/spider-bead-pull/data/adaptive_bead_pull_variations/fits/line_02_gap_02.pkl', '/data/dust/user/salamana/madmax/spider-bead-pull/data/adaptive_bead_pull_variations/fits/line_02_gap_03.pkl', '/data/dust/user/salamana/madmax/spider-bead-pull/

Waiting for 16 fit caches after 1.0 min...


Waiting for 5 fit caches after 2.0 min...


Waiting for 3 fit caches after 3.0 min...


All fit caches are present.


In [7]:
complete, missing = abp.all_fit_caches_exist(
    gaps,
    line_counts=LINE_COUNTS,
    fit_cache_dir=FIT_CACHE_DIR,
    z_ix_options=Z_IX_OPTIONS,
)
assert complete, "Missing fit caches:\n" + "\n".join(str(path) for path in missing[:10])
print("Verified all 16 fit cache files.")


Verified all 16 fit cache files.


## Full-Grid Reference

In [8]:
full_reference = abp.compute_full_reference(
    gaps,
    cache_dir=FULL_CACHE_DIR,
    n_baseline_margin=N_BASELINE_MARGIN,
    frequency_indices=FREQUENCY_INDICES,
)

print("ETs_full", full_reference["ETs_full"].shape)
print("z_full", full_reference["z_full"].shape)
print("frequencies", full_reference["frequencies"].shape)

ETs_full (17, 1001)
z_full (17,)
frequencies (1001,)


## Assemble All 48 Configurations

In [9]:
metrics_rows, metrics_path = abp.assemble_all_configurations(
    gaps=gaps,
    fit_cache_dir=FIT_CACHE_DIR,
    full_reference=full_reference,
    result_root=RESULTS_ROOT,
    figs_root=FIGS_ROOT,
    line_counts=LINE_COUNTS,
    z_ix_options=Z_IX_OPTIONS,
    gap_counts=GAP_COUNTS,
    make_figures=MAKE_FIGURES,
    frequency_indices=FREQUENCY_INDICES,
)

assert len(metrics_rows) == 48, f"Expected 48 metric rows, got {len(metrics_rows)}"

variations_et_path = abp.save_et_results_variations(
    result_root=RESULTS_ROOT,
    output_path=ET_VARIATIONS_PATH,
    line_counts=LINE_COUNTS,
    z_ix_options=Z_IX_OPTIONS,
    gap_counts=GAP_COUNTS,
    gaps=gaps,
)

with np.load(variations_et_path, allow_pickle=False) as variations_et:
    assert variations_et["schema_version"].shape == (1,)
    assert int(variations_et["schema_version"][0]) == 3
    assert variations_et["ETs_reduced"].ndim == 3
    assert variations_et["ETs_reduced"].shape[0] == 48
    assert variations_et["ETs_reduced"].shape[2] == variations_et["frequencies"].shape[0]
    assert variations_et["z_reduced"].shape == variations_et["ETs_reduced"].shape[:2]
    assert variations_et["z_count"].shape == (48,)
    assert variations_et["configuration_lookup"].shape == (48, 4)
    assert variations_et["z_ixs_used"].shape[0] == 48
    assert variations_et["z_ixs_used"].ndim == 3
    assert variations_et["z_ixs_used_count"].shape[0] == 48
    assert variations_et["selected_gap_ranks"].shape[0] == 48
    assert variations_et["selected_gap_ranks_count"].shape == (48,)

variations_et_path


Configurations:   0%|          | 0/48 [00:00<?, ?config/s]

Configurations:   2%|▏         | 1/48 [00:07<05:36,  7.17s/config]

Configurations:   4%|▍         | 2/48 [00:15<06:10,  8.05s/config]

Configurations:   6%|▋         | 3/48 [00:26<06:48,  9.08s/config]

Configurations:   8%|▊         | 4/48 [00:40<08:03, 11.00s/config]

...

Configurations:  96%|█████████▌| 46/48 [10:22<00:29, 14.86s/config]

Configurations:  98%|█████████▊| 47/48 [10:41<00:16, 16.32s/config]

Configurations: 100%|██████████| 48/48 [11:08<00:00, 19.34s/config]

Configurations: 100%|██████████| 48/48 [11:08<00:00, 13.92s/config]

PosixPath('/data/dust/user/salamana/madmax/spider-bead-pull/data/ET_results_variations.npz')

## Quick Metric Preview

In [10]:
for row in metrics_rows[:5]:
    print(row)

print(f"metrics saved to: {metrics_path}")
print(f"configuration figures: {FIGS_ROOT / 'configs'}")
print(f"comparison figures: {FIGS_ROOT / 'comparisons'}")
print(f"ET variation results saved to: {variations_et_path}")

{'line_count': 1, 'z_slice_count': 1, 'gap_count': 1, 'n_reduced_points': 21, 'n_reduced_z': 1, 'fit_success_fraction': 0.999000999000999, 'fit_cost_median': 11491.422350897936, 'fit_cost_p90': 25119.793634115882, 'et_complex_nrmse': 0.5892724850289853, 'rel_abs_et_median': 0.37555334098598825, 'rel_abs_et_p90': 0.7525057696784784, 'et_complex_nrmse_19_20ghz': 0.29384770159886364, 'rel_abs_et_median_19_20ghz': 0.21731333050632604, 'rel_abs_et_p90_19_20ghz': 0.5543850558518126}
{'line_count': 1, 'z_slice_count': 1, 'gap_count': 2, 'n_reduced_points': 21, 'n_reduced_z': 2, 'fit_success_fraction': 0.9995004995004995, 'fit_cost_median': 18389.81949702113, 'fit_cost_p90': 33491.030791429315, 'et_complex_nrmse': 46.28866374986828, 'rel_abs_et_median': 0.38500730520686377, 'rel_abs_et_p90': 0.863133037710225, 'et_complex_nrmse_19_20ghz': 0.29712560668431853, 'rel_abs_et_median_19_20ghz': 0.20022096911933907, 'rel_abs_et_p90_19_20ghz': 0.6079730353441427}
{'line_count': 1, 'z_slice_count': 1, 